## Data Ingestion

### document datastructure

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the content of the document, I am using to create RAG system.",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Kabir Khan",
        "date_created":"2026-04-22"
        }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Kabir Khan', 'date_created': '2026-04-22'}, page_content='This is the content of the document, I am using to create RAG system.')

In [3]:
### create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

### Python Txt File

In [4]:
import os

sample_text = {
    "../data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language...""",
}

for filepath, content in sample_text.items():
    # Extract the directory path (e.g., "data/text_files/")
    folder_path = os.path.dirname(filepath)
    
    # Create the directory if it doesn't exist
    if folder_path:
        os.makedirs(folder_path, exist_ok=True)
        
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


### Machine Learning Txt File

In [5]:
import os

sample_text = {
    "../data/text_files/machine_learning.txt": """Machine learning is a subset of artificial intelligence...""",
}

for filepath, content in sample_text.items():
    # Extract the directory path (e.g., "data/text_files/")
    folder_path = os.path.dirname(filepath)
    
    # Create the directory if it doesn't exist
    if folder_path:
        os.makedirs(folder_path, exist_ok=True)
        
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)
print("Sample text files created successfully.")


Sample text files created successfully.


In [6]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()
print(documents)

c:\Users\Admin\Desktop\RAG_system_working\rag_system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language...')]


In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

### Load all .txt files in the specified directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

document = dir_loader.load()
document

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine learning is a subset of artificial intelligence...'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming language...')]

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154217-05'00'", 'page': 0}, page_content='MoZEUS \nApp for \nEvents'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154

In [9]:
type(pdf_documents[0])

langchain_core.documents.base.Document

## RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [10]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    '''Process all pdf in the directroy'''
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")   
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal documents processed: {len(all_documents)}")
    return all_documents

#Process all PDF's in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")
            


Found 2 PDF files to process

Processing: data.pdf
 Loaded 12 pages

Processing: example.pdf
 Loaded 5 pages

Total documents processed: 17


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154217-05'00'", 'page': 0, 'source_file': 'data.pdf', 'file_type': 'pdf'}, page_content='MoZEUS \nApp for \nEvents'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:2024071

In [13]:
### Text Splitting get into Chucks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    '''Spliting Documents into smaller chunks'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show a example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [14]:
chunks= split_documents(all_pdf_documents)
chunks

Split 17 documents into 33 chunks

Example chunk:
Content: MoZEUS 
App for 
Events...
Metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154217-05'00'", 'page': 0, 'source_file': 'data.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154217-05'00'", 'page': 0, 'source_file': 'data.pdf', 'file_type': 'pdf'}, page_content='MoZEUS \nApp for \nEvents'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:2024071

### Embedding and Vector DB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os
from langchain_core.messages import HumanMessage

In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentanceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager
        Args: 
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of text
        
        Args: 
            texts: List of text strings to embed
            
        Returns: 
            numpy array of embeddings with shape (len(text), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embedding with shape: {embeddings.shape}")
        return embeddings
    
## Initialize the embedding manager
embedding_manager=EmbeddingManager() 
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\Admin\Desktop\RAG_system_working\rag_system\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Model loaded sucessfully. Embedding dimension: 384


### Vector Store

In [17]:
class VectorStore:
    """Manage document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._intialize_store()
        
    def _intialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddign for RAG"}
            )
            print(f"Vector store initialize. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self , documents:List[Any], embeddings: np.ndarray):
        """
        Add documents and their embedding to the vector store
        
        Args:
            documents: List of Langchain documents
            embedding: Correspondings embedding for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of document must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = [] 
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document Content
            documents_text.append(doc.page_content)
            
            # Embedding 
            embeddings_list.append(embedding.tolist())
        
        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore=VectorStore()
vectorstore

Vector store initialize. Collection: pdf_documents
Existing documents in collection: 33


In [18]:
chunks

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154217-05'00'", 'page': 0, 'source_file': 'data.pdf', 'file_type': 'pdf'}, page_content='MoZEUS \nApp for \nEvents'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': '..\\data\\pdf\\data.pdf', 'file_path': '..\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:2024071

In [19]:
### Convert the text into embeddings
texts = [doc.page_content for doc in chunks]
texts

## Generate the Embeddings

embeddings = embedding_manager.generate_embedding(texts)

## Store into Vector Database
vectorstore.add_documents(chunks, embeddings)
print("Total docs in store:", vectorstore.collection.count())

Generating embedding for 33 texts..


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.82s/it]

Generated Embedding with shape: (33, 384)
Adding 33 documents to vector store...
Successfully added 33 documents to vector store
Total documents in collection: 66
Total docs in store: 66


## Retriever Pipeline from Vector Store

In [20]:
class RAGRetriever:
    """Handles query based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embedding
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int= 5, score_threshold:float = 0.0) -> List[Dict[str,Any]]:
        """
        Retreive relavant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generating embedding for query
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0] 
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (chromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                         'id': doc_id,
                         'content': document,
                         'metadata': metadata,
                         'similarity_score': similarity_score,
                         'distance': distance,
                         'rank': i+ 1                            
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever=RAGRetriever(vectorstore, embedding_manager)
rag_retriever
        
        

In [21]:
rag_retriever.retrieve("Hey, how HUBC announce an interim cash dividend")

Retrieving documents for query: 'Hey, how HUBC announce an interim cash dividend'
Top K: 5, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.12it/s]

Generated Embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ed0dac98_13',
  'content': 'en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, \ndue to higher oil prices and growing auto segment contributions from both HUBC and \nNPL, ii) a 22%YoY decline in finance cost as short-term borrowing continues to unwind. \nOn an individual basis, HUBC is expected to report EPS of PkR8.91 (+5%YoY/+9%QoQ), \nwhile NPL is expected at PkR2.37 (+29%YoY/+2.2x QoQ). The YoY increase for NPL is due \ninclusion of EV subsidiary, while higher other income is anticipated is amid elevated cash \nand short term investments.  \nWe anticipate HUBC to announce an interim cash payout of PkR5.0/sh during 3QFY26E, \nwhile no dividend is expected from NPL during the quarter. \nTopline to remain stable in 3QFY26E: Cumulative net revenues for the universe are pro-\njected at PkR20.2bn (+8%YoY/+13%QoQ), as the sequential improvement primarily \nattributable to elevated RFO-based dispatch by both NPL and Narowal. HUBC’s consoli-',


## Integration Vectordb Context Pipeline with LLM output

In [ ]:
### Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM 
groq_api_key = ""

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=2000)

### Simple RAG Function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # retreive the context
    results = retriever.retrieve(query, top_k= top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relavant content found to answer the question"
    # Generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.
        Content:
        {context}

        Question: {query}
        Answer:"""
        
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


In [23]:
answer = rag_simple("What are your analysis about HUBC", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What are your analysis about HUBC'
Top K: 3, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.53it/s]

Generated Embedding with shape: (1, 384)
Retrieved 3 documents (after filtering)


We have a 'HOLD' stance on HUBC with a target price of PkR215/sh for Dec'26. The key factors driving this analysis include improved cash streams from the energy portfolio, stronger-than-expected popularity of BYD's auto segment, and a faster-than-anticipated drawdown in trade debts and short-term borrowings.


### Enhanced RAG Pipeline Features

In [24]:
print("Documents in vector store:", vectorstore.collection.count())

Documents in vector store: 66


In [25]:
print("PDF documents loaded:", len(all_pdf_documents))

PDF documents loaded: 17


In [26]:
print("Chunks created:", len(chunks))

Chunks created: 33


In [27]:
if vectorstore.collection.count() > 0:
    peek = vectorstore.collection.peek(3)
    print("Sample stored docs:", peek['documents'])
else:
    print("❌ Vector store is EMPTY — ingestion never completed")

Sample stored docs: ['MoZEUS \nApp for \nEvents', 'ObjectiveToday we will dis', 'What is MoZEUS?\nEvents are organized initiatives aimed at boosting sales, increasing \nfoot traffic, and attracting customers to your location. Utilize the \nMoZEUS app at your events to effectively engage customers, \ncapturing their information to receive exclusive Cricket offers that \nencourage return visits to your store. \nThe MoZEUS app serves as a marketing tool during events, \ncapturing leads such as customer email and phone numbers. It \nfacilitates interactions by offering customers opportunities to enter \nfor a chance to win $500 and encourages store visits by allowing \nthem to book appointments through the app.']


In [28]:
import os

# ── See exactly where your notebook thinks it is ──────────────────────
print("Current working directory:", os.getcwd())

# ── List what's actually in ../data/ ──────────────────────────────────
data_path = os.path.join(os.getcwd(), "..", "data")
print("Contents of ../data/:", os.listdir(data_path))

Current working directory: c:\Users\Admin\Desktop\RAG_system_working\rag_system\notebook
Contents of ../data/: ['pdf', 'text_files', 'vector_store']


In [29]:
# ── Use absolute path so there's no ambiguity ─────────────────────────
import os
pdf_dir = os.path.abspath("../data/pdf")   # adjust if your folder is named differently
print("Looking for PDFs in:", pdf_dir)
print("Files found:", os.listdir(pdf_dir))

# Re-ingest
all_pdf_documents = process_all_pdfs(pdf_dir)
chunks = split_documents(all_pdf_documents)
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
vectorstore.add_documents(chunks, embeddings)

print("✅ Total docs now in store:", vectorstore.collection.count())

Looking for PDFs in: c:\Users\Admin\Desktop\RAG_system_working\rag_system\data\pdf
Files found: ['data.pdf', 'example.pdf']
Found 2 PDF files to process

Processing: data.pdf
 Loaded 12 pages

Processing: example.pdf
 Loaded 5 pages

Total documents processed: 17
Split 17 documents into 33 chunks

Example chunk:
Content: MoZEUS 
App for 
Events...
Metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-11T15:42:17-05:00', 'source': 'c:\\Users\\Admin\\Desktop\\RAG_system_working\\rag_system\\data\\pdf\\data.pdf', 'file_path': 'c:\\Users\\Admin\\Desktop\\RAG_system_working\\rag_system\\data\\pdf\\data.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'MoZEUS', 'author': 'bonnie kuang', 'subject': '', 'keywords': '', 'moddate': '2024-07-11T15:42:17-05:00', 'trapped': '', 'modDate': "D:20240711154217-05'00'", 'creationDate': "D:20240711154217-05'00'", 'page': 0, 'source_file': 'data.pdf', 'file_type': 'pdf'}
Generating e

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.48it/s]

Generated Embedding with shape: (33, 384)
Adding 33 documents to vector store...
Successfully added 33 documents to vector store
Total documents in collection: 99
✅ Total docs now in store: 99


In [30]:
# ── First, see what scores your query actually returns ─────────────────
results = rag_retriever.retrieve(
    "How much is HUBC to announce an interim cash payout",
    top_k=5,
    score_threshold=0.0   # no filter — see raw scores
)
for r in results:
    print(f"Score: {r['similarity_score']:.4f} | {r['content'][:100]}")

Retrieving documents for query: 'How much is HUBC to announce an interim cash payout'
Top K: 5, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.75it/s]

Generated Embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)
Score: 0.4714 | en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, 
due to higher 
Score: 0.4714 | en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, 
due to higher 
Score: 0.4714 | en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, 
due to higher 
Score: 0.4068 | improved cash streams from the energy portfolio, ii) stronger-than-expected popularity 
of BYD's aut
Score: 0.4068 | improved cash streams from the energy portfolio, ii) stronger-than-expected popularity 
of BYD's aut


In [31]:
# ── Bypass RAGRetriever entirely — query ChromaDB directly ─────────────
test_embedding = embedding_manager.generate_embedding(["HUBC dividend"])[0]

raw_results = vectorstore.collection.query(
    query_embeddings=[test_embedding.tolist()],
    n_results=3
)

print("Raw distances:", raw_results['distances'])
print("Raw documents:", [d[:100] for d in raw_results['documents'][0]])

Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.51it/s]

Generated Embedding with shape: (1, 384)
Raw distances: [[0.4845348596572876, 0.4845348596572876, 0.4845348596572876]]
Raw documents: ['en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, \ndue to higher ', 'en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, \ndue to higher ', 'en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.3bn, \ndue to higher ']


In [32]:
# ── Check if the collection object is still alive ──────────────────────
print("Count:", vectorstore.collection.count())
print("Collection name:", vectorstore.collection.name)

# ── Peek at stored docs to confirm they exist ──────────────────────────
peek = vectorstore.collection.peek(2)
print("Stored doc sample:", peek['documents'])
print("Embedding dims stored:", len(peek['embeddings'][0]))  # should be 384

Count: 99
Collection name: pdf_documents
Stored doc sample: ['MoZEUS \nApp for \nEvents', 'ObjectiveToday we will dis']
Embedding dims stored: 384


In [33]:
# ── Wipe and recreate with cosine distance ─────────────────────────────
vectorstore.client.delete_collection("pdf_documents")

vectorstore.collection = vectorstore.client.get_or_create_collection(
    name="pdf_documents",
    metadata={
        "hnsw:space": "cosine"  # proper 0-1 similarity scores
    }
)

# Re-ingest ONCE
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
vectorstore.add_documents(chunks, embeddings)
print("✅ Count after clean ingest:", vectorstore.collection.count())
# Should be ~30, not 528

Generating embedding for 33 texts..


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.29it/s]

Generated Embedding with shape: (33, 384)
Adding 33 documents to vector store...
Successfully added 33 documents to vector store
Total documents in collection: 33
✅ Count after clean ingest: 33


In [34]:
# After recreating the collection, point retriever to the new one
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [35]:
results = rag_retriever.retrieve("HUBC dividend", top_k=5, score_threshold=0.0)
for r in results:
    print(f"Score: {r['similarity_score']:.4f} | {r['content'][:80]}")
# You should now see varied scores like 0.45, 0.38, 0.31 etc.

Retrieving documents for query: 'HUBC dividend'
Top K: 5, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.10it/s]

Generated Embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)
Score: 0.5155 | en by i) 11%YoY/8%QoQ surge in share of profits from HUBC subsidiaries to PkR11.
Score: 0.4706 | improved cash streams from the energy portfolio, ii) stronger-than-expected popu
Score: 0.4643 | ed to commence in 4QFY26E. The BYD franchise is the largest EV company in Pakist
Score: 0.4589 | AKD Securities Limited 
AKD RESEARCH 
• 
We expect earnings for the AKD power un
Score: 0.3985 | 0% 
Share of Profit 
11,348 
10,195 
11% 
10,492 
8% 
32,634 
30,357 
8% 
Financ


In [36]:
# RAG Advanced Pipeline
from langchain_core.messages import HumanMessage

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page':   doc['metadata'].get('page', 'unknown'),
        'score':  doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}
Answer:"""

    # ✅ Fix: HumanMessage, no .format() or () call
    response = llm.invoke([HumanMessage(content=prompt)])

    output = {'answer': response.content, 'sources': sources, 'confidence': confidence}
    if return_context:
        output['context'] = context
    return output

# Example Usage
result = rag_advanced("Tell me about MoZeus app what does it do and how does it work i want in details answer", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Tell me about MoZeus app what does it do and how does it work i want in details answer'
Top K: 3, Score threshold: 0.1
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.24it/s]


Generated Embedding with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: The MoZEUS app is a marketing tool designed to boost sales and increase foot traffic at events. Here's a detailed explanation of its features and functionality:

**Key Features:**

1. **Lead Capture:** The app captures customer information, including email and phone numbers, to receive exclusive offers and promotions.
2. **Exclusive Offers:** Customers can receive Cricket offers that encourage return visits to the store.
3. **Interactive Contests:** Customers can enter for a chance to win $500, creating a sense of excitement and engagement.
4. **Appointment Booking:** Customers can book appointments through the app, facilitating store visits and increasing sales.
5. **Event Engagement:** The app facilitates interactions between customers and the store, creating a seamless experience.

**How it Works:**

1. **Download and Installation:** The MoZEUS app can be downloaded onto iPads or demo phones usi